## Lab 1. MLP

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing



#daecarga
houses = fetch_california_housing(as_frame=True)
df = houses.frame

### 1. Dataset

In [ ]:
print("=== EXPLORACION Y PREPARACION DATASET ===")
print("\nDimensiones: ")
print(f" - Filas: {df.shape[0]}")
print(f" - Columnas: {df.shape[1]}")
print(f"\n{df.info()}")
print(f"\nEstadisticas:\n{df.describe()}")
print(f"\nVariable objetivo: {houses.target_names[0]}")
print(f"Variable feature: {list(houses.feature_names)}")

null_counts = df.isnull().sum()
print(f"\nValores nulos: {null_counts[null_counts > 0].to_dict() if any(null_counts > 0) else 'No hay valores nulos'}")
duplicates = df.duplicated().sum()
print(f"Valores duplicados: {duplicates}")

#outliers/atipicos con IQR
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    return {
        'col': col,
        'lower': lower,
        'upper': upper,
        'n_outliers': len(outliers),
        'total': len(outliers),
        'pct': len(outliers) / len(df) * 100
    }

print("Valores atipicos/variable:")
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        if info['n_outliers'] > 0:
            print(f"   - {col}: {info['n_outliers']} outliers ({info['pct']:.1f}%)")
        else:
            print(f"   - {col}: No outliers")




=== EXPLORACION Y PREPARACION DATASET ===

Dimensiones: 
 - Filas: 20640
 - Columnas: 9
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB

None

Estadisticas:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.

In [ ]:
def tratamiento_outlier(df, column):
    info = detect_outliers_iqr(df, column)
    n_outliers = info['total']
    pct_outliers = info['pct']
    
    print(f"\n--- {column} ---")
    print(f"Outliers: {n_outliers} ({pct_outliers:.1f}%)")
    print(f"Límite inferior: {info['lower']:.3f}")
    print(f"Límite superior: {info['upper']:.3f}")
    
    #valores extremos
    min_val = df[column].min()
    max_val = df[column].max()

    print(f"Rango: [{min_val:.2f}, {max_val:.2f}]")
    if pct_outliers == 0:
        rec = "No hay outliers detectados"
    elif pct_outliers < 1:
        rec = "Eliminar (pocos outliers, <1% del total)"
    elif pct_outliers < 3:
        rec = "Evaluar según modelo (1-3% del total)"
    elif pct_outliers < 5:
        rec = "Mantener o winsorizar (3-5% del total)"
    else:
        rec = "Mantener (más del 5%, es parte de la distribución)"

    return rec

for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        tratamiento_outlier(df, col)

outliers_info = []
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    if col != 'MedHouseVal':
        info = detect_outliers_iqr(df, col)
        outliers_info.append(info)

#prom de %
pct_outliers_total = np.mean([info['pct'] for info in outliers_info])
print(f"Porcentaje promedio de outliers: {pct_outliers_total:.1f}%")

if pct_outliers_total < 1:
    decision = "Eliminar atipicos (son pocos y no afectan significativamente)"
elif pct_outliers_total < 3:
    decision = "Mantener"
elif pct_outliers_total < 5:
    decision = "Mantener atipicos (son parte de la distribución real)"
else:
    decision = "Mantener atipicos (más del 5%, no son outliers, es la naturaleza de los datos)"

print(f"\nDecisión final: {decision}")


--- MedInc ---
Outliers: 681 (3.3%)
Límite inferior: -0.706
Límite superior: 8.013
Rango: [0.50, 15.00]

--- HouseAge ---
Outliers: 0 (0.0%)
Límite inferior: -10.500
Límite superior: 65.500
Rango: [1.00, 52.00]

--- AveRooms ---
Outliers: 511 (2.5%)
Límite inferior: 2.023
Límite superior: 8.470
Rango: [0.85, 141.91]

--- AveBedrms ---
Outliers: 1424 (6.9%)
Límite inferior: 0.866
Límite superior: 1.240
Rango: [0.33, 34.07]

--- Population ---
Outliers: 1196 (5.8%)
Límite inferior: -620.000
Límite superior: 3132.000
Rango: [3.00, 35682.00]

--- AveOccup ---
Outliers: 711 (3.4%)
Límite inferior: 1.151
Límite superior: 4.561
Rango: [0.69, 1243.33]

--- Latitude ---
Outliers: 0 (0.0%)
Límite inferior: 28.260
Límite superior: 43.380
Rango: [32.54, 41.95]

--- Longitude ---
Outliers: 0 (0.0%)
Límite inferior: -127.485
Límite superior: -112.325
Rango: [-124.35, -114.31]
Porcentaje promedio de outliers: 2.7%

Decisión final: MANTENER o WINSORIZAR (depende del modelo a usar)


**¿Cuántas observaciones y cuántas variables tiene el dataset?** Total de servaciones y 9 variables\
**¿Qué representa cada variable (feature) y cuál es la variable objetivo (target)?**
La variable objetivo es MedHouseVal\
 Fatures:   
 - Longitude: que tan al oeste esta la casa (mientras mas alto el valor más al oeste)
 - Latitude: que tan al norte esta la casa (mientras mas alto el valor más al norte)
 - HouseAge: Edad de una casa en la cuadra
 - AveRooms: promedio de cuartos en la cuadra
 - AveBedrms: promedio de habitaciones en la cuadra
 - Population: total de poblacion en la cuadra
 - AveOccup: promedio de personas por casa
 - MedInc: Media de ingresos por casa por cuadra

**¿Hay valores nulos, duplicados o atípicos (outliers)? ¿Cómo los trató?**

**¿Qué variables son numéricas y cuáles categóricas? ¿Cómo codificó las categóricas?**

**¿Fue necesario normalizar o escalar las variables numéricas?**


### 2. Exploracion y preparación de datos